# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [1]:
import os
import pymongo
import test_solution

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [4]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI')
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi('1'))
# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [5]:
# print all collections in the database
client.list_database_names()

['sample_mflix', 'admin', 'local']

In [6]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21358 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

In [5]:
# ============================================================================
# Question 1
# ============================================================================
def war_movies(movies_collection):
    """
    Enter Your Solution Here
    """
    results = []
    
    # Start your solutions
    search_steps = [
        {
            "$match": {"plot" : {"$regex" : "^war"}}
        },
        {
            "$sort": { "released" : -1}
        },
        {
            "$limit" : 5
        },
        {
            "$project": {"_id": 0, "title": 1, "plot": 1, "released": 1 }
        }
    ]
    
    for mv in movies_collection.aggregate(search_steps):
        results.append({"title": mv["title"], "plot":mv["plot"], "released": mv["released"]})

    # same as: list(movies.find({"plot" : {"$regex" : "war"}}, { "title":1, "plot":1, "released":1, "_id": 0}).sort(["releaased", pymongo.DESCENDING]).limit(50))
    # Your solution ends here
    
    return results


In [16]:
# see if any released field exists
movies.find_one({"plot": {"$regex": "war"}}, {"released": 1})

{'_id': ObjectId('573a1390f29313caabcd5a93'),
 'released': datetime.datetime(1916, 6, 2, 0, 0)}

In [8]:
## Test Your Function Here
war_movies(movies)

# mongo does not return the attribute if it is not present.
# list(movies.find({"plot" : {"$regex" : "war"}}, { "title":1, "plot":1, "released":1, "_id": 0}).sort("released", pymongo.DESCENDING).limit(50))


[]

#### Validating My Solution

In [9]:
# ============================================================================
# Checking Question 1
# ============================================================================
passed, failed, results = test_solution.TestQuestion1.run_all_tests(war_movies, movies)


QUESTION 1: War Movies Query
✅ PASSED: Function exists
✅ PASSED: Returns list
❌ FAILED: Returns 5 or fewer results
   Error: Should return at least 1 result
✅ PASSED: Correct fields returned
✅ PASSED: Plots start with 'war'
✅ PASSED: Sorted by released (ascending)
✅ PASSED: No non-war plots
❌ FAILED: Compare with correct query
   Error: Should return 5 results, got 0
RESULTS: 6 passed, 2 failed


### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [ ]:
# ============================================================================
# Question 2
# ============================================================================
def group_by_rated(movies_collection):
    """
    Group by rated and count the number of movies in each.
    """
    results = []
    pipeline = [
        {"$group": {"_id" : "$rated", "count" : { "$sum" : 1}}}
    ]

    results = list(movies.aggregate(pipeline))
    # End your solution
    
    return results

In [21]:
## Test Your Function Here
group_by_rated(movies)

[{'_id': 'TV-PG', 'count': 76},
 {'_id': 'TV-G', 'count': 59},
 {'_id': 'PASSED', 'count': 181},
 {'_id': None, 'count': 9903},
 {'_id': 'TV-MA', 'count': 60},
 {'_id': 'PG', 'count': 1852},
 {'_id': 'TV-14', 'count': 89},
 {'_id': 'R', 'count': 5537},
 {'_id': 'Not Rated', 'count': 1},
 {'_id': 'OPEN', 'count': 1},
 {'_id': 'PG-13', 'count': 2321},
 {'_id': 'GP', 'count': 44},
 {'_id': 'Approved', 'count': 5},
 {'_id': 'APPROVED', 'count': 709},
 {'_id': 'TV-Y7', 'count': 3},
 {'_id': 'G', 'count': 477},
 {'_id': 'M', 'count': 37},
 {'_id': 'AO', 'count': 3}]

#### Validation My Solution

In [19]:
# ============================================================================
# Checking Question 2
# ============================================================================
passed, failed, results = test_solution.TestQuestion2.run_all_tests(group_by_rated, movies)

QUESTION 2: Group by Rated
✅ PASSED: Function exists
✅ PASSED: Returns list
✅ PASSED: Returns results
❌ FAILED: Result structure
   Error: Each result should have 'movie_count' field
❌ ERROR: Counts are positive
   Error: 'movie_count'
❌ ERROR: Total count matches
   Error: 'movie_count'
✅ PASSED: Includes common ratings
❌ ERROR: Compare with correct implementation
   Error: 'movie_count'
RESULTS: 4 passed, 4 failed



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


In [12]:
# ============================================================================
# Question 3
# ============================================================================
def count_movies_with_comments(movies_collection):
    """
    Count the number of movies with 3 comments or more.
    """
    count = 0
    
    # Start your solution
    pipeline = [
        {"$lookup" :
         {
            "from": "comments",
            "localField": "_id",
            "foreignField": "movie_id",
            "as": "comments"
         }
        },
        { "$match" : { "$expr": { "$gte": [ { "$size": "$comments" }, 3 ] } } },
        { "$count" : "count" }
    ]

    results = list(movies_collection.aggregate(pipeline))
    if results:
        count = results[0]["count"]
    # End your solution
    
    return count

In [20]:
## Test Your Function Here
count_movies_with_comments(movies)

ExecutionTimeout: PlanExecutor error during aggregation :: caused by :: operation exceeded time limit, full error: {'ok': 0.0, 'errmsg': 'PlanExecutor error during aggregation :: caused by :: operation exceeded time limit', 'code': 50, 'codeName': 'MaxTimeMSExpired', '$clusterTime': {'clusterTime': Timestamp(1771814912, 2), 'signature': {'hash': b'\xd0\xd8\x93&\n]E\xbf\x15\x08\xdf\x13V\xb6V\xdc\xe4\x1e\x00\x15', 'keyId': 7581962804794490882}}, 'operationTime': Timestamp(1771814912, 2)}

In [13]:
comments = db.comments
pipeline = [
    {
        "$group": {
            "_id": "$movie_id",
            "num_comments": { "$sum": 1 }
        }
    },
    {
        "$match": {
            "num_comments": { "$gte": 3 }
        }
    },
    {
        "$count": "count"
    }
]

list(comments.aggregate(pipeline))

[{'count': 400}]

In [14]:

pipeline = [
    {
        "$group": {
            "_id": "$movie_id",
            "num_comments": {"$sum": 1}
        }
    },
    {
        "$match": {
            "num_comments": {"$gte": 3}
        }
    },
    {
        "$lookup": {
            "from": "movies",
            "localField": "_id",
            "foreignField": "_id",
            "as": "movie"
        }
    },
    {"$unwind": "$movie"},
    {
        "$project": {
            "_id": 0,
            "title": "$movie.title"
        }
    }
]

results = list(db.comments.aggregate(pipeline))
print("no of movies with >= 3 comments: ", len(results))

no of movies with >= 3 comments:  385


In [17]:
# ============================================================================
# Checking Question 3
# ============================================================================
passed, failed, results = test_solution.TestQuestion3.run_all_tests(count_movies_with_comments, movies)

QUESTION 3: Count Movies with 3+ Comments
✅ PASSED: Function exists
✅ PASSED: Returns integer
✅ PASSED: Returns positive count
✅ PASSED: Count is reasonable
✅ PASSED: Compare with correct query
✅ PASSED: Boundary case (exactly 3)
RESULTS: 6 passed, 0 failed


In [11]:
# check orphan comments
orphan_comments = db.comments.aggregate([
  { "$group": { "_id": "$movie_id", "c": { "$sum": 1 } } },
  { "$match": { "c": { "$gte": 3 } } },
  { "$lookup": { "from": "movies", "localField": "_id", "foreignField": "_id", "as": "m" } },
  { "$match": { "m": { "$size": 0 } } }
])

list(orphan_comments)

[{'_id': ObjectId('573a1397f29313caabce8b6d'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13e1f29313caabdbbd9c'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13a3f29313caabd0df51'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13a7f29313caabd1aa20'), 'c': 150, 'm': []},
 {'_id': ObjectId('573a1395f29313caabce1c43'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a1399f29313caabcec48d'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13c1f29313caabd66006'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13d6f29313caabd9f729'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a1392f29313caabcdbc89'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13a6f29313caabd18366'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a1394f29313caabcdfa27'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13c7f29313caabd73834'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13b2f29313caabd3ad5e'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13a7f29313caabd1b635'), 'c': 3, 'm': []},
 {'_id': ObjectId('573a13acf29313caabd274ca'), 'c': 3, 'm': []}]

In [18]:
# explain the slow join of movie and comments
pipeline = [
    {
        "$lookup": {
            "from": "comments",
            "localField": "_id",
            "foreignField": "movie_id",
            "as": "comments"
        }
    },
    {
        "$match": {
            "$expr": {
                "$gte": [ { "$size": "$comments" }, 3 ]
            }
        }
    },
    { "$count": "count" }
]

explain = movies.database.command({
    "explain": {
        "aggregate": "movies",
        "pipeline": pipeline,
        "cursor": {}
    },
    # "verbosity": "executionStats"
    "verbosity": "queryPlanner"
})

print(explain)


{'explainVersion': '2', 'stages': [{'$cursor': {'queryPlanner': {'namespace': 'sample_mflix.movies', 'parsedQuery': {}, 'indexFilterSet': False, 'queryHash': '6FFA0038', 'planCacheShapeHash': '6FFA0038', 'planCacheKey': '97DBCCA8', 'optimizationTimeMillis': 0, 'maxIndexedOrSolutionsReached': False, 'maxIndexedAndSolutionsReached': False, 'maxScansToExplodeReached': False, 'prunedSimilarIndexes': False, 'winningPlan': {'isCached': False, 'queryPlan': {'stage': 'EQ_LOOKUP', 'planNodeId': 3, 'foreignCollection': '6991658a9ac14aa006b55ee8_sample_mflix.comments', 'localField': '_id', 'foreignField': 'movie_id', 'asField': 'comments', 'strategy': 'NestedLoopJoin', 'scanDirection': 'forward', 'inputStage': {'stage': 'PROJECTION_SIMPLE', 'planNodeId': 2, 'transformBy': {'_id': True}, 'inputStage': {'stage': 'COLLSCAN', 'planNodeId': 1, 'filter': {}, 'direction': 'forward'}}}, 'slotBasedPlan': {'slots': '$$RESULT=s18 env: {  }', 'stages': '[3] mkobj s18 s4 [] drop [comments = s17] true false \n